## PCA + Metadata

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

from scipy.stats import ttest_ind

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"


### Carregando o dataset

In [3]:
DATASET_PATH = "../data/processed/dataset_features.parquet"

df_dataset = pd.read_parquet(DATASET_PATH)
df_dataset.shape

(165, 393)

In [4]:
#  Age, Gender and condition
# Create a dictionary where keys are the metric names (e.g., 'degree')
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_"
]

NODE_FEATURES = [
    c for c in df_dataset.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]
df_dataset = df_dataset.rename(columns={
    "Gender_ 1=female_2=male": "gender",
    "Age": "age",
    "Handedness": "handedness",
    "Education": "education",
    "DRUG_0=negative_1=Positive": "drug",
    "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
    "SKID_Diagnoses": "skid_diagnoses",
    "SKID_Diagnoses 1": "skid_diagnoses_1",
    "SKID_Diagnoses 2": "skid_diagnoses_2",
    "Comments_SKID_assessment": "comments_skid_assessment",
    "Hamilton_Scale": "hamilton_scale",
    "BSL23_sumscore": "bsl23_sumscore",
    "BSL23_behavior": "bsl23_behavior",
    "AUDIT": "audit",
    "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
    "Alcohol_Dependence_In_1st-3rd_Degree_relative": "alcohol_dependence_in_1st_3rd_degree_relative",
    "Relationship_Status": "relationship_status"
})

In [5]:

X = df_dataset[NODE_FEATURES]
meta = df_dataset[["condition", "age", "gender"]]


imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
pca = PCA(n_components=2)

X_imputed = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imputed)
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
df_pca = pd.concat([df_pca, meta.reset_index(drop=True)], axis=1)
df_pca

c:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['hub_frequency_0' 'hub_frequency_1' 'hub_frequency_2' 'hub_frequency_3'
 'hub_frequency_4' 'hub_frequency_5' 'hub_frequency_6' 'hub_frequency_7'
 'hub_frequency_8' 'hub_frequency_9' 'hub_frequency_10' 'hub_frequency_11'
 'hub_frequency_12' 'hub_frequency_13' 'hub_frequency_14'
 'hub_frequency_15' 'hub_frequency_16' 'hub_frequency_17'
 'hub_frequency_18' 'hub_frequency_19' 'hub_frequency_20'
 'hub_frequency_21' 'hub_frequency_22' 'hub_frequency_23'
 'hub_frequency_24' 'hub_frequency_25' 'hub_frequency_26'
 'hub_frequency_27' 'hub_frequency_28' 'hub_frequency_29'
 'hub_frequency_30' 'hub_frequency_31' 'hub_frequency_32'
 'hub_frequency_33' 'hub_frequency_34' 'hub_frequency_35'
 'hub_frequency_36' 'hub_frequency_37' 'hub_frequency_38'
 'hub_frequency_39' 'hub_frequency_40' 'hub_frequency_41'

,PC1,PC2,condition,age,gender
0,7.004574,-0.838805,EO,30-35,2
1,3.100349,0.403037,EC,25-30,2
2,-2.178715,-0.511710,EO,25-30,2
3,-7.659837,1.832979,EC,25-30,1
4,5.779680,-0.175907,EO,25-30,1
...,...,...,...,...,...
160,4.058802,-3.388772,EO,20-25,2
161,10.493644,0.035598,EC,20-25,2
162,2.472713,-0.827225,EO,20-25,2
163,6.782618,-2.149128,EC,25-30,2


In [6]:

color_vars = ["condition", "age", "gender"]

for var in color_vars:
    fig = px.scatter(
        df_pca,
        x="PC1",
        y="PC2",
        color=var,
        title=f"PCA colored by {var}",
        opacity=0.8,
        template="plotly_dark"
    )
    fig.show()
